In [1]:
import pymc as pm

In [2]:
import arviz as az

In [5]:
import numpy as np
import pandas as pd
import pandas as pd
import numpy as np
import torch
from torch import nn

In [7]:
import numpy as np
import pandas as pd

data = pd.read_csv("/Users/syeda/MobiWatch/dataset/mobiflow/new.csv", sep=';')
#print(data.head())

In [8]:
print(data.dtypes)

msg_type                  str
msg_id                  int64
ts                    float64
ver                       str
gen                       str
bs_id                   int64
rnti                    int64
tmsi                    int64
imsi                    int64
imei                    int64
cipher_alg              int64
int_alg                 int64
est_cause               int64
msg                       str
rrc_state               int64
nas_state               int64
sec_state               int64
emm_cause               int64
rrc_init_timer        float64
rrc_inactive_timer      int64
nas_initial_timer     float64
nas_inactive_timer    float64
Attackornot               str
dtype: object


In [9]:
data['msg'] = data['msg'].astype(str).astype('category')

In [10]:
variable_idx = data['msg'].cat.codes.values
variable_names = data['msg'].cat.categories.values

In [11]:
print(f"\nEncoded categories: {variable_idx}")
print(f"Category Mapping: {dict(enumerate(variable_names))}")


Encoded categories: [11  9 10 ...  0 11  9]
Category Mapping: {0: 'Authenticationrequest', 1: 'Authenticationresponse', 2: 'DeregistrationrequestUEoriginating', 3: 'Identityrequest', 4: 'Identityresponse', 5: 'LoggedMeasurementConfiguration-r16', 6: 'MeasurementReport', 7: 'MobilityFromNRCommand', 8: 'RRCReconfigurationComplete', 9: 'RRCSetup', 10: 'RRCSetupComplete', 11: 'RRCSetupRequest', 12: 'Registrationreject', 13: 'Registrationrequest', 14: 'SecurityModeCommand', 15: 'SecurityModeComplete', 16: 'SecurityModeFailure', 17: 'Securitymodecommand', 18: 'Securitymodecomplete', 19: 'UECapabilityEnquiry', 20: 'ULInformationTransfer'}


In [12]:
data['Attackornot'] = data['Attackornot'].astype(str).astype('category')
target_binary = data['Attackornot'].cat.codes.values 
target_names = data['Attackornot'].cat.categories.values

In [13]:
print(f"Mapping for Target: {dict(enumerate(target_names))}")

Mapping for Target: {0: 'Attack', 1: 'Benign'}


In [14]:
coords = {"Features": variable_names}

with pm.Model(coords=coords) as classification_model:

    beta = pm.Normal("beta", mu=0, sigma=1.5, dims="Features")     
    p = pm.math.invlogit(beta[variable_idx]) 
    group_probs = pm.Deterministic("group_probabilities", pm.math.invlogit(beta), dims="Features")
    y_obs = pm.Bernoulli("y_obs", p=p, observed=target_binary)
    
    idata = pm.sample(draws=1000, tune=1000, return_inferencedata=True)
    print(az.summary(idata, var_names=["group_probabilities"]))

/Users/syeda/radioconda/envs/bayesian/lib/python3.12/site-packages/threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [beta]


Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 14 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


                                                           mean      sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
group_probabilities[Authenticationrequest]                0.786  0.0396     0.72     0.85     4169     1403  1.00   0.00061  0.00041
group_probabilities[Authenticationresponse]              0.9205  0.0282     0.87     0.96     5702     1628  1.00   0.00038  0.00032
group_probabilities[DeregistrationrequestUEoriginating]   0.732   0.185     0.38     0.96     4038     1109  1.00    0.0028    0.002
group_probabilities[Identityrequest]                      0.742   0.144     0.48     0.93     4551     1400  1.00    0.0023   0.0017
group_probabilities[Identityresponse]                     0.662   0.146     0.41     0.87     5514     1653  1.00     0.002   0.0013
group_probabilities[LoggedMeasurementConfiguration-r16]   0.649   0.226     0.24     0.94     5063     1629  1.00    0.0033   0.0019
group_probabilities[MeasurementReport]                    0.727   0.1

In [15]:
print(az.summary(idata, var_names=["group_probabilities"]))

                                                           mean      sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
group_probabilities[Authenticationrequest]                0.786  0.0396     0.72     0.85     4169     1403  1.00   0.00061  0.00041
group_probabilities[Authenticationresponse]              0.9205  0.0282     0.87     0.96     5702     1628  1.00   0.00038  0.00032
group_probabilities[DeregistrationrequestUEoriginating]   0.732   0.185     0.38     0.96     4038     1109  1.00    0.0028    0.002
group_probabilities[Identityrequest]                      0.742   0.144     0.48     0.93     4551     1400  1.00    0.0023   0.0017
group_probabilities[Identityresponse]                     0.662   0.146     0.41     0.87     5514     1653  1.00     0.002   0.0013
group_probabilities[LoggedMeasurementConfiguration-r16]   0.649   0.226     0.24     0.94     5063     1629  1.00    0.0033   0.0019
group_probabilities[MeasurementReport]                    0.727   0.1

In [16]:
print("Posterior variables:")
print(list(idata.posterior.data_vars))

Posterior variables:
['beta', 'group_probabilities']


In [17]:
print(list(idata.posterior.data_vars))

['beta', 'group_probabilities']
